# 实验 10 · 将数据和模型接入 verl

前面已经分别练习了训练、评估和保存。本课把这些组件接到 verl：导出 Hugging Face 模型与 tokenizer（分词器），准备 Parquet 数据和奖励函数，再用当前安装的 verl 加载数据并检查训练配置。

本课的检查范围是文件格式、数据接口与配置解析。完整的 Ray/vLLM 训练将在第 11 课使用另一套微型模型与配置验证。 模型是第六课的小字符模型，奖励纯数字答案；不具备通用聊天能力。

## 1 · 补全可保存的字符 tokenizer

保留第六课的原始 token ID，按单字符切分，并验证编码、解码和 chat template 与训练时的提示一致。聊天模板只拼接内容，适合本教学单轮任务，不是生产多轮对话模板。新增未知字符 token 后扩展嵌入，但已训练 token ID 不变。

In [ ]:
import json,sys,subprocess,importlib.util
from pathlib import Path
import pandas as pd
import torch
from tokenizers import Tokenizer,models,pre_tokenizers,decoders
from transformers import PreTrainedTokenizerFast,Qwen2ForCausalLM,AutoTokenizer
from omegaconf import OmegaConf

source=Path('support/experiment06_output')
record=json.loads((source/'results.json').read_text())
vocab=record['vocab']
out=Path('support/experiment10_output').resolve();out.mkdir(exist_ok=True)
model_dir=out/'model'
# 模型的嵌入表用整数取行。沿用第 06 课的字符编号，才能让同一字符在导出、重载后仍取到同一行权重。
mapping={c:i for i,c in enumerate(vocab)};mapping['[UNK]']=len(mapping)
raw=Tokenizer(models.WordLevel(mapping,unk_token='[UNK]'))
raw.pre_tokenizer=pre_tokenizers.Split(pattern='',behavior='isolated')
raw.decoder=decoders.Fuse()
tokenizer=PreTrainedTokenizerFast(tokenizer_object=raw,pad_token='_',eos_token='#',unk_token='[UNK]')
tokenizer.chat_template="{% for message in messages %}{{ message['content'] }}{% endfor %}"
for text in ['3+7=','03+07=10#','69+69=138#']:
    ids=tokenizer.encode(text,add_special_tokens=False)
    assert ids==[mapping[c] for c in text]
    assert tokenizer.decode(ids,skip_special_tokens=False)==text
tokenizer.save_pretrained(model_dir)
model=Qwen2ForCausalLM.from_pretrained(source/'model')
model.resize_token_embeddings(len(tokenizer),mean_resizing=False)
model.save_pretrained(model_dir)
loaded=AutoTokenizer.from_pretrained(model_dir)
assert loaded.apply_chat_template([dict(role='user',content='3+7=')],tokenize=True,add_generation_prompt=True)==[mapping[c] for c in '3+7=']
print('PASS：token ID、解码、chat template 和保存重载一致。')

## 2 · 创建真实 Parquet 与奖励函数

使用第六课训练数对的前 32 题和验证数对的前 8 题作为接口检查样本，不使用最终测试题。每行包含 `prompt` 消息列表、`data_source` 和 `reward_model.ground_truth`。

奖励函数接受 verl 的调用参数。它只能读取正确答案用于评分，不能在生成提示中泄漏答案。这里是严格数字协议，输出两种候选答案或解释文本均不得分。

In [ ]:
# 一行 Parquet 对应一道题。prompt 是消息列表，进入模型上下文；reward_model.ground_truth 单独留给评分函数。
# 例如 (3,7) 的消息内容是“3+7=”，参考答案是字符串“10”。extra_info.index 用于追踪样本。
def make_rows(pairs):
    return [dict(data_source='tutorial_addition',prompt=[dict(role='user',content=f'{a}+{b}=')],
                 reward_model=dict(style='rule',ground_truth=str(a+b)),extra_info=dict(index=i)) for i,(a,b) in enumerate(pairs)]
train=record['train_pairs'][:32];validation=record['val_pairs'][:8]
assert not {tuple(sorted(p)) for p in train}&{tuple(sorted(p)) for p in validation}
pd.DataFrame(make_rows(train)).to_parquet(out/'train.parquet')
pd.DataFrame(make_rows(validation)).to_parquet(out/'val.parquet')
# 这里生成供 verl 动态加载的 Python 文件。solution_str 是解码后的回答，ground_truth 来自数据行；返回的浮点数就是该回答的奖励。
reward_source=chr(10).join(['def compute_score(data_source, solution_str, ground_truth, extra_info=None):','    return float(solution_str.strip() == str(ground_truth))',''])
(out/'reward.py').write_text(reward_source,encoding='utf-8')
spec=importlib.util.spec_from_file_location('tutorial_reward',out/'reward.py')
reward=importlib.util.module_from_spec(spec);spec.loader.exec_module(reward)
assert reward.compute_score('tutorial_addition','10','10')==1
assert reward.compute_score('tutorial_addition','100','10')==0
assert reward.compute_score('tutorial_addition','10 or 12','10')==0
print('PASS：Parquet 导出、划分与奖励函数测试通过。')

## 3 · 用当前 verl 数据集类读取

导入 verl 的 `RLHFDataset` 和 `DataProto`，检查数据批次。当前安装版本把 prompt 的最终分词放在 rollout AgentLoop 中，所以 dataset 返回 `raw_prompt`，不要误以为每个版本都直接返回 token 张量。

还会用导出的 tokenizer 和模型做一次 GPU 生成，确认接口能连接。单次答案是否正确与接口是否可运行是两件事。

In [ ]:
from verl.utils.dataset.rl_dataset import RLHFDataset,collate_fn
from verl import DataProto
# OmegaConf 把这些键值组成配置对象。max_prompt_length 按 token 计；开启过滤后，超长提示可能让数据集行数变少。
cfg=OmegaConf.create(dict(max_prompt_length=32,filter_overlong_prompts=True,filter_overlong_prompts_workers=1,cache_dir=str(out/'cache')))
# 传入文件路径、刚重载的 tokenizer 和读取配置。dataset[i] 取一条经过 verl 数据接口处理的记录。
dataset=RLHFDataset(str(out/'train.parquet'),loaded,cfg)
# 先取 4 条记录，collate_fn 按字段把它们合并；from_single_dict 再把批量字段封装成 verl 的 DataProto。
# len(dataset) 数整份数据集的题，len(batch) 数这次取出的题。这里预期分别是 32 和 4。
batch=DataProto.from_single_dict(collate_fn([dataset[i] for i in range(4)]))
assert len(dataset)==32 and len(batch)==4
print('真实 verl 数据批次:',len(batch),'条；字段:',list(dataset[0]))
prompt=dataset[0]['raw_prompt']
# 先按模板把消息转成模型输入，再返回 PyTorch 张量。tokenize=True 表示返回 token ID，cuda() 把张量移到 GPU。
tokens=loaded.apply_chat_template(prompt,tokenize=True,add_generation_prompt=True,return_tensors='pt').cuda()
model=model.cuda().eval()
with torch.inference_mode():
    response=model.generate(tokens,attention_mask=torch.ones_like(tokens),max_new_tokens=4,do_sample=False,
        pad_token_id=loaded.pad_token_id,eos_token_id=loaded.eos_token_id)
# generate 返回提示和新生成的 token。切掉前 tokens.shape[1] 个位置后，评分函数才只看到模型的回答。
answer=loaded.decode(response[0,tokens.shape[1]:],skip_special_tokens=True)
truth=dataset[0]['reward_model']['ground_truth']
print('题目:',prompt,'答案:',repr(answer),'奖励:',reward.compute_score('tutorial_addition',answer,truth))
del model;torch.cuda.empty_cache()

## 4 · 加载真实训练配置，但不启动训练

使用 `--cfg job` 只生成配置，验证参数名能被当前 verl 接受。模板设为单 GPU、短序列、小批量，本课导出的模型与这份配置尚未验证完整训练，因此去掉 `--cfg job` 后仍需单独排查资源和运行问题。第 11 课提供的是另一套经过验证的微型训练配置。

actor 是更新策略的模型，rollout 是生成过程，ref 是 KL 参考策略；GRPO 的组内优势不需要 PPO 那样的独立 value critic。本课不启用 actor KL，因此没有验证 ref 工作进程。

第 11 课会先完成一次更新，检查显存、参数变化和检查点；第 12 课再加入任务奖励与固定评估。

In [ ]:
# 每个字符串都是一项 Hydra 配置覆盖，格式为“配置路径=值”。先读左侧模块名，再读右侧取值。
overrides=[
    # algorithm 选优势估计方法。GRPO 用同题多个回答的奖励作比较；train_files、val_files 分别指向训练与验证题。
    'algorithm.adv_estimator=grpo',f'data.train_files={out}/train.parquet',f'data.val_files={out}/val.parquet',
    # 一个训练批取 4 道提示。提示长度和回答长度分别限制，单位都是 token。
    'data.train_batch_size=4','data.max_prompt_length=16','data.max_response_length=4',
    # model.path 是模型和 tokenizer 的来源。actor 是接受更新的策略，rollout 是负责生成候选的组件，这里交给 vLLM。
    f'actor_rollout_ref.model.path={model_dir}','actor_rollout_ref.rollout.name=vllm',
    # 张量并行度 1 表示一次推理使用一个分片。n=4 表示每道提示生成 4 个候选；4 道提示对应 16 条回答。
    'actor_rollout_ref.rollout.tensor_model_parallel_size=1','actor_rollout_ref.rollout.n=4',
    # 推理引擎的 max_model_len 约束提示与回答的总长度。gpu_memory_utilization 是 vLLM 的显存预算比例。
    'actor_rollout_ref.rollout.max_model_len=32','actor_rollout_ref.rollout.gpu_memory_utilization=0.2',
    # mini-batch 决定策略更新时如何分组；micro-batch 是每张卡一次前向和反向处理的更小分块，用来控制峰值显存。
    'actor_rollout_ref.actor.ppo_mini_batch_size=4','actor_rollout_ref.actor.ppo_micro_batch_size_per_gpu=1',
    # false 关闭 actor 损失中的参考策略 KL 项。后面两项声明训练资源：每个节点一张 GPU，共一个节点。
    'actor_rollout_ref.actor.use_kl_loss=false','trainer.n_gpus_per_node=1','trainer.nnodes=1',
    # 若实际启动训练，步数上限是 1，日志写到控制台。本格下面使用 --cfg job，只检查配置能否成功组合。
    'trainer.total_training_steps=1','trainer.logger=[console]',
    # path 告诉 verl 去哪里找评分代码；name 指定该文件中要调用的函数。任一名称写错，都可能在实际加载时失败。
    f'custom_reward_function.path={out}/reward.py','custom_reward_function.name=compute_score']
command=[sys.executable,'-m','verl.trainer.main_ppo','--cfg','job']+overrides
result=subprocess.run(command,text=True,capture_output=True,timeout=120)
(out/'configuration_check.log').write_text(result.stdout+result.stderr,encoding='utf-8')
assert result.returncode==0,result.stderr
(out/'config_arguments.json').write_text(json.dumps(command,indent=2),encoding='utf-8')
print('PASS：真实 verl 配置加载成功，未启动训练。')
print('完整输出:',out/'configuration_check.log')

## 5 · 基础结业检查

运行结果检查能证明文件和接口存在；下面的理解题需要你自己回答，不等于系统替你通过了理解考核。

1. GPU 可用为什么不等于所有代码自动加速？
2. SFT 的监督信号与 GRPO 的奖励/优势有什么区别？
3. 一组回答全对/全错时，标准化组内优势有什么性质？
4. 为何要按数对而不是按文本行划分训练和测试？
5. 为什么只恢复模型权重不保证准确续训？
6. 第七课长加法成功中，哪些规则由程序员提供？
7. 为什么 `--cfg job` 成功不能证明训练成功？

<details><summary>参考答案（先自行回答）</summary>

1. 张量和模型要在 CUDA 设备上执行，数据传输也有开销。
2. SFT 用标准答案 token 的交叉熵；GRPO 对采样回答打分，比较组内相对奖励后更新策略。
3. 奖励相同则优势为零，奖励项没有区分方向。
4. 交换加数和格式变体可能泄漏同一道题的信息。
5. 优化器、随机状态、步数等也影响后续更新。
6. 位对齐、倒序循环、进位传递与类别解码均为人工提供。
7. 配置解析不执行 Ray/vLLM 初始化、生成、反向传播或检查点保存。

</details>

如果你已完成实验并能解释上述问题，就可以进入高级课程，进一步验证 verl 的完整训练流程。基础部分练习的 GPU 计算、训练信号、评估和状态恢复，会继续用于后面的排错。完整分布式训练、LoRA、大模型部署和多 GPU 优化属于后续进阶部分。

In [ ]:
required={
    'GPU experiment':'01_从GPU到verl的第一步.ipynb','SFT':'support/experiment02_output/results.json',
    'RL':'support/experiment03_output/results.json','controls':'support/experiment04_output/results.json',
    'evaluation':'support/experiment05_output/evaluation.json','coverage':'support/experiment06_output/results.json',
    'composition':'support/experiment07_output/results.json','reward audit':'support/experiment08_output/reward_audit.csv',
    'resume':'support/experiment09_output/verification.json'}
checks={key:Path(value).exists() for key,value in required.items()}
checks.update(tokenizer=(model_dir/'tokenizer.json').exists(),parquet=len(dataset)==32,config=result.returncode==0)
display(pd.DataFrame(checks.items(),columns=['artifact_or_check','passed']))
(out/'foundation_checks.json').write_text(json.dumps(checks,indent=2),encoding='utf-8')
assert all(checks.values()),'有前置课程产物缺失，请先完成对应课程'
print('基础实验产物与本课接口检查全部通过；理解题请自行作答。')

---

<a href="https://github.com/SergioTermann"><img src="support/assets/logo.png" alt="SergioTermann" width="44" height="44" align="left" style="margin: 0 12px 0 0;"></a>

**更多有趣的 AI 课程**  
[← 访问 SergioTermann 的 GitHub 主页](https://github.com/SergioTermann)